# Interacting with APIs in Python

APIs (Application Programming Interfaces) let you request data from external services over HTTP.  
This notebook covers the full spectrum — from open public APIs to authenticated ones using **API Keys** and **OAuth**.

| Auth Type | When Used | Examples |
|---|---|---|
| None | Fully public data | Open APIs, some GitHub endpoints |
| API Key | Simple identity verification | OpenWeatherMap, NASA, News APIs |
| OAuth 2.0 | Acting on behalf of a user | Twitter/X, GitHub user actions, Spotify |

## 1. Import Libraries

In [1]:
import requests
import json
import pandas as pd
from requests_oauthlib import OAuth1

print("Libraries ready!")

Libraries ready!


## 2. Calling a Public API (No Authentication)

The simplest kind of API — no credentials needed.  
We'll use the **GitHub API** to fetch public repos for a user.

```
GET https://api.github.com/users/{username}/repos
```

`requests.get()` sends the HTTP request. The response comes back as JSON, which we parse with `.json()`.

In [2]:
url = "https://api.github.com/users/torvalds/repos"

response = requests.get(url)

print(f"Status code : {response.status_code}")   # 200 = OK
print(f"Content-Type: {response.headers['Content-Type']}")

# Parse the JSON response
repos = response.json()

# Turn it into a DataFrame
df_repos = pd.DataFrame(repos)

# Show relevant columns
print(f"\nTotal repos fetched: {len(df_repos)}")
df_repos[["name", "language", "stargazers_count", "forks_count", "updated_at"]].head(10)

Status code : 200
Content-Type: application/json; charset=utf-8

Total repos fetched: 11


,name,language,stargazers_count,forks_count,updated_at
0,1590A,OpenSCAD,546,21,2026-02-28T15:22:30Z
1,AudioNoise,C,4265,200,2026-03-02T20:55:27Z
2,GuitarPedal,C,1796,63,2026-03-02T17:23:40Z
3,HunspellColorize,C,310,12,2026-03-01T07:44:33Z
4,libdc-for-dirk,C,375,51,2026-03-01T07:49:24Z
5,libgit2,C,340,25,2026-03-02T16:45:02Z
6,linux,C,220308,60711,2026-03-02T22:58:25Z
7,pesconvert,C,550,73,2026-03-01T07:56:54Z
8,subsurface-for-dirk,C++,439,67,2026-03-01T07:51:57Z
9,test-tlb,C,950,215,2026-03-02T14:00:15Z


## 3. Using Query Parameters

Most APIs accept **query parameters** to filter or customize the response.  
They are appended to the URL after `?` like `?key=value&key2=value2`.  
With `requests`, pass them as a dictionary to the `params` argument — cleaner and safer.

In [3]:
# Using the Open-Meteo API — free weather API, no key needed
url = "https://api.open-meteo.com/v1/forecast"

params = {
    "latitude": 37.7749,      # San Francisco
    "longitude": -122.4194,
    "current": "temperature_2m,wind_speed_10m,weathercode",
    "temperature_unit": "celsius",
    "wind_speed_unit": "mph",
    "timezone": "America/Los_Angeles"
}

response = requests.get(url, params=params)

# The full URL that was actually sent (params encoded automatically)
print("Request URL:", response.url)
print("Status     :", response.status_code)

data = response.json()
current = data["current"]

print(f"\n📍 San Francisco — Current Weather")
print(f"   Temperature : {current['temperature_2m']} °C")
print(f"   Wind Speed  : {current['wind_speed_10m']} mph")
print(f"   Time        : {current['time']}")

Request URL: https://api.open-meteo.com/v1/forecast?latitude=37.7749&longitude=-122.4194&current=temperature_2m%2Cwind_speed_10m%2Cweathercode&temperature_unit=celsius&wind_speed_unit=mph&timezone=America%2FLos_Angeles
Status     : 200

📍 San Francisco — Current Weather
   Temperature : 16.1 °C
   Wind Speed  : 5.6 mph
   Time        : 2026-03-02T15:00


## 4. API Key Authentication

Many APIs require an **API key** — a token you include in the request to prove your identity.  
It can be passed as a query parameter or in the request **headers** (more secure).

> 🔑 The example below uses the [Open-Meteo geocoding API](https://open-meteo.com/) (still free/keyless), but the pattern for header-based API keys is shown in the comments.  
> Replace `YOUR_API_KEY` with a real key from any service like NASA, OpenWeatherMap, etc.

In [4]:
# --- Pattern 1: API key as a query parameter ---
# response = requests.get(url, params={"api_key": "YOUR_API_KEY", ...})

# --- Pattern 2: API key in the Authorization header (most common & secure) ---
# headers = {"Authorization": "Bearer YOUR_API_KEY"}
# response = requests.get(url, headers=headers)

# --- Pattern 3: Custom header key (e.g. some APIs use "x-api-key") ---
# headers = {"x-api-key": "YOUR_API_KEY"}
# response = requests.get(url, headers=headers)

# Live example: NASA APOD (Astronomy Picture of the Day) with demo key
# NASA provides a "DEMO_KEY" for testing — limited rate, but functional
NASA_API_KEY = "DEMO_KEY"

url = "https://api.nasa.gov/planetary/apod"
params = {"api_key": NASA_API_KEY}

response = requests.get(url, params=params)
data = response.json()

print(f"Status : {response.status_code}")
print(f"\n🚀 NASA Astronomy Picture of the Day")
print(f"   Title : {data['title']}")
print(f"   Date  : {data['date']}")
print(f"   URL   : {data['url']}")
print(f"\n   Explanation (first 300 chars):\n   {data['explanation'][:300]}...")

Status : 200

🚀 NASA Astronomy Picture of the Day
   Title : The Dusty Surroundings of Orion and the Pleiades
   Date  : 2026-03-02
   URL   : https://apod.nasa.gov/apod/image/2603/DustyOrionPleiades_Fernandez_960.jpg

   Explanation (first 300 chars):
   How well do you know the night sky? OK, but how well can you identify famous sky objects in a very deep image? Either way, here is a test: see if you can find some well-known night-sky icons in a deep image filled with filaments of normally faint dust and gas.  This image contains the Pleiades star ...


## 5. OAuth 1.0 Authentication — Python equivalent of the R Twitter example

The R code in the course does this:
```r
myapp <- oauth_app("twitter", key="yourConsumerKeyHere", secret="yourConsumerSecretHere")
sig   <- sign_oauth1.0(myapp, token="yourTokenHere", token_secret="yourTokenSecretHere")
homeTL <- GET("https://api.twitter.com/1.1/statuses/home_timeline.json", sig)
```

In Python the exact equivalent uses **`requests` + `requests-oauthlib`**:

- `consumer_key` / `consumer_secret` → the app credentials (from the developer portal)
- `access_token` / `access_token_secret` → the user-level credentials
- `OAuth1` → signs every request automatically, just like `sign_oauth1.0()` in R

> ⚠️ The Twitter/X API now requires a paid developer account. The code below is correct and ready to use — just replace the placeholder strings with your real credentials.

In [5]:
from requests_oauthlib import OAuth1

# -------------------------------------------------------------------
# R equivalent:
#   myapp <- oauth_app("twitter", key="...", secret="...")
#   sig   <- sign_oauth1.0(myapp, token="...", token_secret="...")
# -------------------------------------------------------------------

# Your app credentials (from developer.twitter.com)
consumer_key        = "yourConsumerKeyHere"
consumer_secret     = "yourConsumerSecretHere"

# Your user-level credentials
access_token        = "yourTokenHere"
access_token_secret = "yourTokenSecretHere"

# Build the OAuth1 signature object — equivalent to sign_oauth1.0()
auth = OAuth1(
    consumer_key,
    consumer_secret,
    access_token,
    access_token_secret
)

# -------------------------------------------------------------------
# R equivalent:
#   homeTL <- GET("https://api.twitter.com/1.1/statuses/home_timeline.json", sig)
# -------------------------------------------------------------------
url = "https://api.twitter.com/1.1/statuses/home_timeline.json"
params = {"count": 5, "tweet_mode": "extended"}

response = requests.get(url, auth=auth, params=params)

print(f"Status: {response.status_code}")

if response.status_code == 200:
    tweets = response.json()
    for i, tweet in enumerate(tweets, 1):
        print(f"\n[{i}] @{tweet['user']['screen_name']}: {tweet['full_text'][:100]}...")
else:
    # Expected if credentials are placeholders
    print("Response:", response.json())

Status: 401
Response: {'errors': [{'code': 89, 'message': 'Invalid or expired token.'}]}


## 6. OAuth 2.0 — Modern Standard (e.g. GitHub with a Personal Access Token)

OAuth 2.0 is simpler — you just pass a **Bearer token** in the `Authorization` header.  
This is what most modern APIs (Spotify, GitHub, Reddit, etc.) use today.

> 🔑 To get a GitHub token: go to **Settings → Developer Settings → Personal Access Tokens**.

In [6]:
# GitHub works without a token for public data (but rate-limited to 60 req/hr)
# With a token you get 5000 req/hr

GITHUB_TOKEN = "yourGitHubTokenHere"   # replace with your real token

headers = {
    "Authorization": f"Bearer {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28"
}

# Without auth still works for public endpoints — let's demonstrate that
url = "https://api.github.com/repos/pandas-dev/pandas"
response = requests.get(url)   # no token needed for public repos

data = response.json()

print(f"Status : {response.status_code}")
print(f"\n📦 Repository: {data['full_name']}")
print(f"   Description : {data['description']}")
print(f"   Stars       : {data['stargazers_count']:,}")
print(f"   Forks       : {data['forks_count']:,}")
print(f"   Open Issues : {data['open_issues_count']:,}")
print(f"   Language    : {data['language']}")
print(f"   Created     : {data['created_at']}")

# Rate limit info is always in the response headers
print(f"\n🔢 Rate Limit  : {response.headers.get('X-RateLimit-Limit')}")
print(f"   Remaining   : {response.headers.get('X-RateLimit-Remaining')}")

Status : 200

📦 Repository: pandas-dev/pandas
   Description : Flexible and powerful data analysis / manipulation library for Python, providing labeled data structures similar to R data.frame objects, statistical functions, and much more
   Stars       : 48,014
   Forks       : 19,713
   Open Issues : 3,686
   Language    : Python
   Created     : 2010-08-24T01:37:33Z

🔢 Rate Limit  : 60
   Remaining   : 59


## 7. Handling API Responses Properly

Always check the status code before processing the response.  
Good practice: raise an exception automatically for any error code (4xx, 5xx).

In [ ]:
def call_api(url, params=None, headers=None, auth=None):
    """
    Safe API caller with error handling.
    Returns parsed JSON or None on failure.
    """
    try:
        response = requests.get(url, params=params, headers=headers, auth=auth, timeout=10)
        response.raise_for_status()   # raises HTTPError for 4xx/5xx responses
        return response.json()

    except requests.exceptions.HTTPError as e:
        print(f"❌ HTTP Error: {e.response.status_code} — {e.response.reason}")
    except requests.exceptions.ConnectionError:
        print("❌ Connection Error: could not reach the server.")
    except requests.exceptions.Timeout:
        print("❌ Timeout: the server took too long to respond.")
    except requests.exceptions.RequestException as e:
        print(f"❌ Unexpected error: {e}")
    return None


# Test with a valid endpoint
data = call_api("https://api.github.com/users/guido-van-rossum")
if data:
    print(f"✅ Success! User: {data['login']}, Repos: {data['public_repos']}")

# Test with a bad endpoint (will trigger 404)
bad = call_api("https://api.github.com/users/thisuserdoesnotexist_xyz_999")
if bad is None:
    print("Handled the error gracefully.")